In [ ]:
# Cell 1 — vLLM Installer (handles both online and offline Kaggle)
import subprocess, sys
from pathlib import Path

try:
    import vllm
    print(f'vLLM {vllm.__version__} already installed.')
except ImportError:
    print("Installing vLLM...")
    
    # Try online first (works in interactive Kaggle sessions)
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', 'vllm', '-q'],
            timeout=300,
        )
        print('vLLM installed (online).')
    except Exception:
        # Offline fallback: Kaggle submission mode
        print("Online install failed, trying offline wheels...")
        
        # Fix numpy for vLLM 0.7.x
        np_wheels = list(Path('/kaggle/input').glob('**/numpy-1.26*cp312*.whl'))
        if np_wheels:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', str(np_wheels[0]), '--no-index', '-q'])
        
        # Install xgrammar first (the missing dep)
        xg_wheels = list(Path('/kaggle/input').glob('**/xgrammar*.whl'))
        if xg_wheels:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', str(xg_wheels[0]), '--no-index', '-q'])
        
        wheel_dirs = list({p.parent for p in Path('/kaggle/input').glob('**/*.whl')})
        args = [sys.executable, '-m', 'pip', 'install', 'vllm', '--no-index', '-q']
        for d in wheel_dirs:
            args.extend(['--find-links', str(d)])
        subprocess.check_call(args)
        print('vLLM installed (offline).')

# Also ensure peft is available for LoRA training
try:
    import peft
    print(f'PEFT {peft.__version__} ready.')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'peft', '-q'])
    print('PEFT installed.')

print('Environment ready.')

In [ ]:
# Cell 2 — Configuration
import gc
import hashlib
import json
import os
import re
import random
import subprocess
import sys
import threading
import time
import traceback
import numpy as np
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from collections import Counter

# --- Paths ---
WORKING = Path('/kaggle/working')
ARC_DATA = None
for p in [
    Path('/kaggle/input/arc-prize-2026/arc-agi-2_training_challenges'),
    Path('/kaggle/input/arc-agi-3/arc-agi-3/data/training'),
    Path('/kaggle/input/arc-prize-2026/arc-agi-3/data/training'),
]:
    if p.exists():
        ARC_DATA = p
        break
if ARC_DATA is None:
    ARC_DATA = WORKING / 'arc_data' / 'training'

DSL_PATH = WORKING / 'dsl.py'
METRIC_FILE = WORKING / 'metric.json'
RESULTS_DIR = WORKING / 'evolution_results'
RESULTS_DIR.mkdir(exist_ok=True)
HYPOTHESES_FILE = RESULTS_DIR / 'hypotheses.jsonl'
SUCCESSFUL_PROGRAMS_PATH = RESULTS_DIR / 'successful_programs.jsonl'

# --- Model ---
# Qwen3-30B-A3B MoE: 30B total, 3B active per token (fits T4x2).
#
# Priority order:
#   1. GPTQ-Int4 from HF (if uploaded to Kaggle as dataset) — fastest, ~24.5GB
#   2. Kaggle official qwen3.5-35b-a3b + bitsandbytes INT4 — no upload needed
#   3. HF model ID (needs internet)
MODEL_PATH = os.environ.get("ARC_MODEL_PATH", "Qwen/Qwen3-30B-A3B-GPTQ-Int4")

# Auto-detect: check Kaggle mounts in priority order
_model_candidates = [
    # Qwen3-30B-A3B GPTQ-Int4 — best fit for T4x2 (~16GB weights)
    ("/kaggle/input/qwen3-30b-a3b-gptq-int4", "gptq"),
    ("/kaggle/input/qwenqwen3-30b-a3b-gptq-int4", "gptq"),
    # Qwen3-30B-A3B Thinking 2507 4-bit
    ("/kaggle/input/qwen3-30b-a3b-thinking-2507-4bit", "gptq"),
    # Qwen3-30B-A3B Instruct 2507 GPTQ
    ("/kaggle/input/junhowieqwen3-30b-a3b-instruct-2507-gptq", "gptq"),
    # Qwen3.5-35B-A3B GPTQ-Int4 (tight on T4x2 — 24.5GB weights)
    ("/kaggle/input/qwen35-35b-a3b-gptq-int4", "gptq"),
    # Official Kaggle model (unquantized — needs bitsandbytes INT4)
    ("/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-35b-a3b/1", "bitsandbytes"),
    ("/kaggle/input/models/qwen-lm/qwen-3/transformers/qwen3-30b-a3b/1", "bitsandbytes"),
]
QUANTIZATION = os.environ.get("ARC_QUANTIZATION", "gptq")  # default for GPTQ

for _path, _quant in _model_candidates:
    if Path(_path).exists():
        MODEL_PATH = _path
        QUANTIZATION = _quant
        break

TP = int(os.environ.get("ARC_TP", "2"))
MAX_NEW_TOKENS = int(os.environ.get("ARC_MAX_TOKENS", "3072"))

# --- Evolution ---
MAX_REFLECTIONS = 3
NUM_CANDIDATES = 10  # vLLM batches efficiently
CODOPT_BRANCHES = 3
TIER3_TASKS = 2
TASKS_PER_DIAGNOSTIC = 5
OUTER_ROUNDS = 20  # Kaggle 12h limit ~ 20 rounds

# --- LoRA ---
LORA_STATE_PATH = RESULTS_DIR / 'lora_state.json'
LORA_ADAPTERS_DIR = RESULTS_DIR / 'lora_adapters'
LORA_MIN_PROGRAMS = 50
LORA_MIN_ROUNDS_BETWEEN = 5
LORA_SKIP_FIRST_ROUNDS = 3
LORA_MAX_KEPT_ADAPTERS = 2  # less disk on Kaggle

print(f"Model:        {MODEL_PATH}")
print(f"TP:           {TP}")
print(f"Quantization: {QUANTIZATION}")
print(f"ARC data:     {ARC_DATA}")
print(f"Max tokens:   {MAX_NEW_TOKENS}")

In [ ]:
# Cell 3 — Copy DSL from dataset + Backend init
import shutil

MODULES = Path('/kaggle/input/modules')
if MODULES.exists():
    shutil.copy(MODULES / 'dsl.py', DSL_PATH)
    print(f'Copied dsl.py ({DSL_PATH.stat().st_size:,} bytes)')
else:
    print(f'WARNING: {MODULES} not found — using existing dsl.py')

# ---------------------------------------------------------------------------
# BACKEND: vLLM with LoRA adapter support
# ---------------------------------------------------------------------------

_llm = None
_active_lora = None  # path to active LoRA adapter, or None


def _init_backend(lora_adapter=None):
    """Initialize vLLM engine. Optionally enable LoRA adapter serving.
    
    Handles two quantization paths:
      - "gptq": pre-quantized GPTQ-Int4 model (fastest, recommended)
      - "bitsandbytes": on-the-fly INT4 quantization of unquantized model
    """
    global _llm, _active_lora
    import torch
    from vllm import LLM

    enable_lora = lora_adapter is not None

    # Build vLLM kwargs based on quantization strategy
    llm_kwargs = dict(
        model=MODEL_PATH,
        tensor_parallel_size=TP,
        gpu_memory_utilization=0.90,
        max_model_len=4096,
        trust_remote_code=True,
        enable_lora=enable_lora,
    )
    if enable_lora:
        llm_kwargs["max_lora_rank"] = 16

    if QUANTIZATION == "gptq":
        # GPTQ-Int4: pre-quantized. Use standard "gptq" (not "gptq_marlin")
        # to avoid ValidationError on models missing Marlin config flags.
        llm_kwargs["dtype"] = "bfloat16"
        llm_kwargs["quantization"] = "gptq"
    elif QUANTIZATION == "bitsandbytes":
        # Unquantized model → on-the-fly INT4 via bitsandbytes.
        # vLLM supports this via quantization="bitsandbytes" with load_format="bitsandbytes".
        llm_kwargs["dtype"] = "float16"
        llm_kwargs["quantization"] = "bitsandbytes"
        llm_kwargs["load_format"] = "bitsandbytes"
    else:
        # Fallback: let vLLM auto-detect
        llm_kwargs["dtype"] = "float16" if torch.cuda.is_available() else "float32"
        if QUANTIZATION and QUANTIZATION != "auto":
            llm_kwargs["quantization"] = QUANTIZATION

    print(f"[backend] Init vLLM: {MODEL_PATH}")
    print(f"  TP={TP} dtype={llm_kwargs.get('dtype')} quant={QUANTIZATION}"
          f"{f' lora={lora_adapter}' if lora_adapter else ''}")

    _llm = LLM(**llm_kwargs)
    _active_lora = lora_adapter
    print("[backend] vLLM ready.")


def unload_model():
    """Free GPU memory before LoRA training."""
    global _llm, _active_lora
    import torch
    if _llm is not None:
        del _llm
        _llm = None
    _active_lora = None
    gc.collect()
    torch.cuda.empty_cache()
    print("[backend] Model unloaded, CUDA cache cleared.")


def reload_with_adapter(adapter_path=None):
    """Reload vLLM engine, optionally with a LoRA adapter."""
    unload_model()
    _init_backend(lora_adapter=adapter_path)


def _generate_batch(prompts, temperature=0.0):
    """Batched generation via vLLM. Returns list of response strings."""
    if _llm is None:
        _init_backend(_active_lora)

    from vllm import SamplingParams
    params = SamplingParams(
        temperature=max(temperature, 1e-6),
        max_tokens=MAX_NEW_TOKENS,
        stop=["```\n\n", "```\n#", "\n\nif __name__"],
    )

    kwargs = {}
    if _active_lora:
        from vllm.lora.request import LoRARequest
        kwargs["lora_request"] = LoRARequest("active", 1, _active_lora)

    outputs = _llm.generate(prompts, params, **kwargs)
    return [o.outputs[0].text for o in outputs]


def call_model(prompt, temperature=0.0, max_tokens=None):
    """Single-prompt wrapper."""
    old_max = MAX_NEW_TOKENS
    if max_tokens:
        globals()['MAX_NEW_TOKENS'] = max_tokens
    try:
        responses = _generate_batch([prompt], temperature)
    finally:
        globals()['MAX_NEW_TOKENS'] = old_max
    text = responses[0] if responses else ""
    print(f"[model] {len(text)} chars. Preview: {repr(text[:120])}", flush=True)
    return text


def generate(prompt, temperature=0.0, max_tokens=None):
    """Alias used by evolution loop."""
    return call_model(prompt, temperature, max_tokens)


print("Backend functions defined.")

In [ ]:
# Cell 4 — ARC Helpers (code extraction, execution, prompts)

HELPER_FUNCTIONS = {"np": np, "deepcopy": deepcopy}


def fix_indentation(code):
    code = code.replace("\t", "    ")
    lines = code.split("\n")
    return "\n".join(line if line.strip() else "" for line in lines)


def extract_python_code(text):
    code = None
    m = re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL)
    if m:
        code = m.group(1)
    if not code:
        m = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
        if m and "def transform" in m.group(1):
            code = m.group(1)
    if not code:
        m = re.search(r"(def\s+transform\s*\(.*)", text, re.DOTALL)
        if m:
            code = m.group(1)
    if not code:
        code = text
    if "def transform" in code:
        code = code[code.find("def transform"):]
        lines = code.splitlines()
        trimmed = []
        for i, line in enumerate(lines):
            if i == 0:
                trimmed.append(line)
                continue
            if line.startswith((" ", "\t")) or line.strip() == "" or line.lstrip().startswith("#"):
                trimmed.append(line)
                continue
            break
        code = "\n".join(trimmed)
    return fix_indentation(code)


def grid_to_str(grid):
    if not grid or not isinstance(grid, list):
        return str(grid)
    try:
        return "\n".join("".join(str(c) for c in row) for row in grid)
    except Exception:
        return str(grid)


def run_with_timeout(fn, args, timeout_sec=5):
    result, error = [None], [None]
    def target():
        try:
            result[0] = fn(*args)
        except Exception as e:
            error[0] = e
    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout_sec)
    if t.is_alive():
        raise TimeoutError("Infinite loop detected")
    if error[0]:
        raise error[0]
    return result[0]


def try_code_on_task(code, task_data, evaluate_on_test=False):
    ns = dict(HELPER_FUNCTIONS)
    failures = []
    pairs = task_data.get("test", []) if evaluate_on_test else task_data.get("train", [])

    try:
        dsl_code = DSL_PATH.read_text()
    except Exception:
        dsl_code = ""

    try:
        exec(dsl_code + "\n" + code, ns)
        transform_fn = ns.get("transform")
        if not transform_fn:
            return False, [(pairs[0]["input"], pairs[0]["output"], None, "No 'transform' function", None)]

        for pair in pairs:
            try:
                pred = run_with_timeout(transform_fn, (pair["input"],), timeout_sec=5)
                pred_list = [list(row) for row in pred] if pred else []
                out_list = [list(row) for row in pair["output"]]
                if pred_list != out_list:
                    failures.append((pair["input"], pair["output"], pred_list, None, None))
            except Exception as e:
                failures.append((pair["input"], pair["output"], None, str(e), traceback.format_exc()))
    except Exception as e:
        tb = traceback.format_exc()
        if pairs:
            failures.append((pairs[0]["input"], pairs[0]["output"], None, str(e), tb))
        else:
            failures.append(("", "", None, str(e), tb))

    return len(failures) == 0, failures


def calculate_pixel_accuracy(expected, predicted):
    if not predicted or not expected:
        return 0.0
    if not isinstance(predicted, list) or not all(isinstance(r, list) for r in predicted):
        return 0.0
    if len(set(len(r) for r in predicted)) > 1:
        return 0.0
    try:
        exp = np.array(expected)
        pred = np.array(predicted)
        max_r = max(exp.shape[0], pred.shape[0])
        max_c = max(exp.shape[1], pred.shape[1])
        canvas_exp = np.full((max_r, max_c), -1.0)
        canvas_pred = np.full((max_r, max_c), -2.0)
        canvas_exp[:exp.shape[0], :exp.shape[1]] = exp
        canvas_pred[:pred.shape[0], :pred.shape[1]] = pred
        base_score = float(np.sum(canvas_exp == canvas_pred) / (max_r * max_c))
        spatial_score, valid_colors = 0.0, 0
        for color in np.unique(canvas_exp):
            if color < 0:
                continue
            coords_exp = np.argwhere(canvas_exp == color)
            coords_pred = np.argwhere(canvas_pred == color)
            if coords_exp.size == 0 or coords_pred.size == 0:
                continue
            diff = coords_exp[:, np.newaxis, :] - coords_pred[np.newaxis, :, :]
            distances = np.linalg.norm(diff, ord=1, axis=2)
            hausdorff = max(np.max(np.min(distances, axis=1)), np.max(np.min(distances, axis=0)))
            spatial_score += max(0.0, 1.0 - hausdorff / (max_r + max_c))
            valid_colors += 1
        bonus = (spatial_score / valid_colors) * 0.5 if valid_colors > 0 else 0.0
        return float(min(1.0, base_score * 0.5 + bonus))
    except Exception:
        return 0.0


def build_prompt(task_data):
    result = [None]
    err = [None]
    def _run():
        try:
            ns = {"grid_to_str": grid_to_str}
            exec(DSL_PATH.read_text(), ns)
            if "build_prompt" in ns:
                result[0] = ns["build_prompt"](task_data)
        except Exception as e:
            err[0] = e
    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout=120)
    if result[0] is not None:
        return result[0]
    if err[0]:
        print(f"[!] DSL prompt error: {err[0]}")
    prompt = "ARC puzzle.\n"
    for i, pair in enumerate(task_data["train"]):
        prompt += f"Ex{i+1} In:\n{grid_to_str(pair['input'])}\nOut:\n{grid_to_str(pair['output'])}\n\n"
    prompt += "Output ONLY python code `def transform(input_grid):`\n```python\n"
    return prompt


def build_task_hints(task_data):
    try:
        full_prompt = build_prompt(task_data)
        if "HINTS:\n" in full_prompt:
            hint_block = full_prompt.split("HINTS:\n", 1)[1]
            if "Output ONLY" in hint_block:
                hint_block = hint_block.split("Output ONLY", 1)[0].rstrip()
            if hint_block:
                return "HINTS:\n" + hint_block + "\n\n"
    except Exception:
        pass
    return ""


def build_reflection_prompt(task_data, code, failures, cached_hints=""):
    prompt = "WRONG. Fix this code:\n```python\n" + code + "\n```\nErrors:\n"
    for fail in failures[:2]:
        inp, expected, got, err_msg = fail[0], fail[1], fail[2], fail[3]
        if err_msg:
            prompt += f"- {err_msg}\n"
        else:
            prompt += f"- Expected:\n{grid_to_str(expected)}\n  Got:\n{grid_to_str(got)}\n"
    prompt += "\nTraining examples:\n"
    for i, pair in enumerate(task_data.get("train", [])[:3]):
        prompt += f"Ex{i+1} In:\n{grid_to_str(pair['input'])}\nOut:\n{grid_to_str(pair['output'])}\n\n"
    prompt += cached_hints
    prompt += (
        "Revise the rule so it matches every training example. "
        "Prefer short helper-based code and preserve rectangular list-of-lists output.\n"
        "Output ONLY corrected code. NO explanation.\n```python\ndef transform(input_grid):\n"
    )
    return prompt


def _log_successful_program(task_name, task_data, code, prompt):
    entry = {
        "task_name": task_name, "timestamp": time.time(),
        "code": code, "prompt": prompt,
        "num_train": len(task_data.get("train", [])),
        "num_test": len(task_data.get("test", [])),
    }
    try:
        with open(SUCCESSFUL_PROGRAMS_PATH, "a") as f:
            f.write(json.dumps(entry) + "\n")
    except Exception:
        pass


def hunter_seeker_arc_jsons(root_path, max_tasks):
    valid_tasks = [
        p for p in root_path.rglob("*.json")
        if "metadata" not in p.name and "package" not in p.name
    ]
    random.shuffle(valid_tasks)
    return valid_tasks[:max_tasks]


print("ARC helpers defined.")

In [ ]:
# Cell 5 — Three-Tier Evaluation + Diagnostic Prompts

CORE_FUNCTIONS = [
    "detect_background_color", "get_objects", "shape", "palette",
    "color_counts", "copy_grid", "rotate_cw", "mirror_h",
]

RESEARCH_CONTEXT = """\
Key ARC-AGI solution techniques:
1. CONNECTED COMPONENTS: Identify connected regions of same-colored cells.
2. FLOOD FILL: Fill enclosed regions bounded by a specific color.
3. SYMMETRY DETECTION: Detect reflective and rotational symmetry.
4. GRID DECOMPOSITION: Split grids into subgrids, quadrants, or tiles.
5. COLOR MAPPING: Systematic color replacement or remapping rules.
6. OBJECT MANIPULATION: Extract, move, scale, or compose objects.
7. PATTERN COMPLETION: Extend or complete partially shown patterns.
8. BOUNDARY/EDGE OPERATIONS: Detect borders, extract outlines."""


def load_dsl_namespace():
    ns = {"grid_to_str": grid_to_str}
    exec(DSL_PATH.read_text(), ns)
    return ns


def _prepare_helper_ns(ns):
    helper_code = ns.get("HELPER_CODE_PREFIX", "")
    helper_ns = {"__builtins__": __builtins__, "np": np, "deepcopy": deepcopy}
    try:
        exec(helper_code, helper_ns)
        return helper_ns, None
    except Exception as e:
        return None, str(e)


def eval_task_tier1(helper_ns, task_data):
    grid = task_data["train"][0]["input"]
    passed = 0
    for fname in CORE_FUNCTIONS:
        fn = helper_ns.get(fname)
        if fn is None:
            continue
        try:
            result = run_with_timeout(fn, (grid,), timeout_sec=2)
            if result is not None:
                passed += 1
        except Exception:
            pass
    return passed / len(CORE_FUNCTIONS)


def eval_task_tier2(ns, task_data):
    bp = ns.get("build_prompt")
    if bp is None:
        return 0.0
    try:
        result = run_with_timeout(bp, (task_data,), timeout_sec=5)
        if result is None or len(str(result)) < 50:
            return 0.0
        return 1.0
    except Exception:
        return 0.0


def _fallback_prompt(task_data):
    prompt = "ARC puzzle.\n"
    for i, pair in enumerate(task_data["train"]):
        prompt += f"Ex{i+1} In:\n{grid_to_str(pair['input'])}\nOut:\n{grid_to_str(pair['output'])}\n\n"
    prompt += "Output ONLY python code `def transform(input_grid):`\n```python\n"
    return prompt


def _truncate_traceback(tb, max_lines=8):
    lines = tb.strip().split("\n")
    if len(lines) <= max_lines:
        return tb.strip()
    return "...\n" + "\n".join(lines[-max_lines:])


def solve_task_single(task_data, task_name="unknown"):
    """Tier 3: One-shot solve via vLLM. Returns (pixel_accuracy, traces)."""
    ns = load_dsl_namespace()
    bp = ns.get("build_prompt")
    if bp:
        try:
            prompt = run_with_timeout(bp, (task_data,), timeout_sec=10)
        except Exception:
            prompt = None
        if not prompt:
            prompt = _fallback_prompt(task_data)
    else:
        prompt = _fallback_prompt(task_data)

    response = generate(prompt, temperature=0.0, max_tokens=1024)
    code = extract_python_code(response)
    if not code or "def transform" not in code:
        return 0.0, [{"task_name": task_name, "failure_type": "NO_TRANSFORM",
                       "error_message": f"LLM response had no transform ({len(response)} chars)"}]

    passed, failures = try_code_on_task(code, task_data)
    if passed:
        _log_successful_program(task_name, task_data, code, prompt)
        return 1.0, []

    traces = []
    for fail in failures:
        inp, expected, predicted, err_msg = fail[0], fail[1], fail[2], fail[3]
        tb = fail[4] if len(fail) > 4 else None
        ftype = "CRASH" if predicted is None else "WRONG_ANSWER"
        pa = calculate_pixel_accuracy(expected, predicted) if predicted else 0.0
        traces.append({
            "task_name": task_name, "failure_type": ftype,
            "error_message": err_msg[:200] if err_msg else None,
            "stack_trace": _truncate_traceback(tb) if tb else None,
            "pixel_accuracy": pa,
            "input_shape": (len(inp), len(inp[0])) if isinstance(inp, list) and inp else (0, 0),
            "output_shape": (len(expected), len(expected[0])) if isinstance(expected, list) and expected else (0, 0),
            "predicted_shape": (len(predicted), len(predicted[0])) if isinstance(predicted, list) and predicted else None,
        })
    best_pa = max(t["pixel_accuracy"] for t in traces) if traces else 0.0
    return best_pa, traces


def find_failing_tasks(sample_size=50, tier3_count=None, seed=42):
    if tier3_count is None:
        tier3_count = TIER3_TASKS

    ns = load_dsl_namespace()
    helper_ns, helper_err = _prepare_helper_ns(ns)
    if helper_ns is None:
        print(f"[evolve] WARNING: Helper exec failed: {helper_err}")
        helper_ns = {}

    task_files = list(ARC_DATA.glob("*.json"))
    if not task_files:
        print(f"[evolve] No tasks found in {ARC_DATA}")
        return []

    random.seed(seed)
    sample = random.sample(task_files, min(sample_size, len(task_files)))

    scored = []
    for tf in sample:
        with open(tf) as f:
            task_data = json.load(f)
        t1 = eval_task_tier1(helper_ns, task_data)
        t2 = eval_task_tier2(ns, task_data)
        combined = t1 * 0.6 + t2 * 0.4
        scored.append((tf, task_data, t1, t2, combined))

    scored.sort(key=lambda x: x[4])
    failing_tuples = [(tf, td, t1, t2, c) for tf, td, t1, t2, c in scored if c < 1.0]
    perfect = len(scored) - len(failing_tuples)
    print(f"[evolve] Tier 1+2: {len(failing_tuples)} imperfect, {perfect} perfect out of {len(sample)}")

    results = []

    if failing_tuples:
        tier3_targets = failing_tuples[:tier3_count]
        if tier3_targets:
            print(f"[evolve] Tier 3: solving {len(tier3_targets)} worst tasks...")
            for tf, td, t1, t2, combined in tier3_targets:
                try:
                    solve_score, traces = solve_task_single(td, task_name=tf.stem)
                except Exception as e:
                    solve_score, traces = 0.0, [{"task_name": tf.stem, "failure_type": "CRASH", "error_message": str(e)}]
                category = "CRASH" if t1 < 1.0 else ("PROMPT_FAIL" if t2 < 1.0 else "WRONG_ANSWER")
                results.append({
                    "path": tf, "task_data": td, "category": category,
                    "tier1": t1, "tier2": t2, "tier3": solve_score, "combined": combined,
                    "traces": traces,
                })
                print(f"  [tier3] {tf.name}: {category} t1={t1:.2f} t2={t2:.2f} t3={solve_score:.2f}")

        tier3_paths = {r["path"] for r in results}
        for tf, td, t1, t2, combined in failing_tuples:
            if tf not in tier3_paths:
                category = "CRASH" if t1 < 1.0 else ("PROMPT_FAIL" if t2 < 1.0 else "UNTESTED")
                results.append({
                    "path": tf, "task_data": td, "category": category,
                    "tier1": t1, "tier2": t2, "tier3": None, "combined": combined,
                })
    else:
        tier3_targets = scored[:tier3_count]
        print(f"[evolve] Tier 1+2 all perfect. Tier 3: solving {len(tier3_targets)} tasks...")
        for tf, td, t1, t2, combined in tier3_targets:
            try:
                solve_score, traces = solve_task_single(td, task_name=tf.stem)
            except Exception as e:
                solve_score, traces = 0.0, [{"task_name": tf.stem, "failure_type": "CRASH", "error_message": str(e)}]
            results.append({
                "path": tf, "task_data": td, "category": "SOLVE_FAIL",
                "tier1": t1, "tier2": t2, "tier3": solve_score, "combined": combined,
                "traces": traces,
            })
            print(f"  [tier3] {tf.name}: SOLVE_FAIL t1={t1:.2f} t2={t2:.2f} t3={solve_score:.2f}")

        results = [r for r in results if r.get("tier3") is not None and r["tier3"] < 1.0]
        results.sort(key=lambda r: r["tier3"])
        if results:
            print(f"[evolve] Found {len(results)} tier3 failures out of {len(tier3_targets)} evaluated")

    return results


def compute_solve_score(failing_results):
    tier3_scores = [r["tier3"] for r in failing_results if r["tier3"] is not None]
    if not tier3_scores:
        return 0.0
    return sum(tier3_scores) / len(tier3_scores)


# --- Diagnostic prompt templates ---

DIAGNOSTIC_PROMPT = """\
/no_think
You are an ARC-AGI DSL engineer. Given failing tasks, output ONLY Python functions. No analysis, no explanation — just code.

Current DSL: {num_functions} functions. Do NOT duplicate existing functions.

## Known Techniques
{research_context}

## Existing (last 50 signatures, do NOT duplicate)
{existing_signatures}

## Previous Experiments ([-] = no improvement, [+] = improved)
{hypothesis_history}

## BLACKLIST — DO NOT propose functions similar to these:
{blacklist}

## Failure Distribution: {category_summary}

## Failing Tasks
{task_descriptions}

## Execution traces from failed solve attempts
{execution_traces}

IMPORTANT: Study the input/output pairs carefully. Each task has a UNIQUE transformation rule.

OUTPUT EXACTLY 3-5 new Python functions in ```python blocks. Each function:
- Takes `grid: list[list[int]]` as first arg, returns `list[list[int]]`
- Is self-contained (only stdlib + numpy)
- Does ONE transformation relevant to the failing tasks above
- Has a one-line docstring

```python
def function_name(grid: list[list[int]]) -> list[list[int]]:
    \"\"\"One-line description.\"\"\"
    import numpy as np
    # implementation
    return result
```

START WITH ```python IMMEDIATELY. No preamble."""


def _extract_function_signatures(helper_code):
    return re.findall(r"(def \w+\([^)]*\))", helper_code)


def _format_hypothesis_history(entries):
    if not entries:
        return "No previous experiments."
    lines = []
    for e in entries:
        improved = e.get("metric_after", 0) > e.get("metric_before", 0)
        icon = "+" if improved else "-"
        funcs = ", ".join(e.get("proposed_functions", [])[:3])
        lines.append(f"  [{icon}] Round {e.get('round', '?')}: {e.get('hypothesis', '')[:80]} ({funcs})")
    return "\n".join(lines)


def _build_blacklist(entries):
    tried = set()
    for e in entries:
        for fname in e.get("proposed_functions", []):
            words = fname.replace("_", " ").split()
            if len(words) >= 2:
                tried.add("_".join(words[:3]))
    return sorted(tried)


def _count_stagnant_streak(entries):
    streak = 0
    for e in reversed(entries):
        if e.get("status", "stagnant") != "keep":
            streak += 1
        else:
            break
    return streak


def load_recent_hypotheses(n=10):
    if not HYPOTHESES_FILE.exists():
        return []
    lines = HYPOTHESES_FILE.read_text().strip().split("\n")
    entries = []
    for line in lines[-n:]:
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return entries


def build_diagnostic_prompt(failing_results):
    ns = load_dsl_namespace()
    helper_code = ns.get("HELPER_CODE_PREFIX", "")
    num_functions = helper_code.count("\ndef ") + 1

    sigs = _extract_function_signatures(helper_code)
    sig_lines = [f"  {s}" for s in sigs[-50:]]
    if len(sigs) > 50:
        sig_lines.insert(0, f"  ... ({len(sigs) - 50} more) ...")
    sig_text = "\n".join(sig_lines) if sig_lines else "  (none)"

    recent = load_recent_hypotheses(10)
    history_text = _format_hypothesis_history(recent)
    blacklist = _build_blacklist(recent)
    blacklist_text = ", ".join(blacklist) if blacklist else "(none yet)"

    shown = [r for r in failing_results if r.get("tier3") is not None][:TASKS_PER_DIAGNOSTIC]
    if not shown:
        shown = failing_results[:TASKS_PER_DIAGNOSTIC]

    categories = {}
    for r in failing_results:
        categories[r["category"]] = categories.get(r["category"], 0) + 1
    category_summary = ", ".join(f"{k}: {v}" for k, v in sorted(categories.items()))

    task_descs = []
    for r in shown:
        td = r["task_data"]
        desc = f"### Task: {r['path'].name} [{r['category']}]\n"
        desc += f"Scores: tier1={r['tier1']:.2f} tier2={r['tier2']:.2f}"
        if r.get("tier3") is not None:
            desc += f" tier3(solve)={r['tier3']:.2f}"
        desc += "\n"
        for i, pair in enumerate(td["train"][:2]):
            inp, out = pair["input"], pair["output"]
            desc += f"Train {i+1} Input ({len(inp)}x{len(inp[0])}):\n{grid_to_str(inp)}\n"
            desc += f"Train {i+1} Output ({len(out)}x{len(out[0])}):\n{grid_to_str(out)}\n"
        task_descs.append(desc)

    trace_descs = []
    for r in shown:
        for t in r.get("traces", [])[:2]:
            tdesc = f"- {t.get('task_name', '?')}: {t['failure_type']}"
            if t.get("error_message"):
                tdesc += f"\n    Error: {t['error_message']}"
            if t.get("stack_trace"):
                tdesc += f"\n    Traceback:\n      {t['stack_trace'].replace(chr(10), chr(10) + '      ')}"
            if t["failure_type"] == "WRONG_ANSWER":
                tdesc += f"\n    Pixel accuracy: {t.get('pixel_accuracy', 0):.2f}"
            trace_descs.append(tdesc)

    return DIAGNOSTIC_PROMPT.format(
        num_functions=num_functions,
        research_context=RESEARCH_CONTEXT.strip(),
        existing_signatures=sig_text,
        hypothesis_history=history_text,
        blacklist=blacklist_text,
        category_summary=category_summary,
        task_descriptions="\n".join(task_descs),
        execution_traces="\n".join(trace_descs) if trace_descs else "(no traces captured)",
    )


def get_prompt_score():
    try:
        with open(METRIC_FILE) as f:
            return json.load(f).get("prompt_score", None)
    except Exception:
        return None


print("Evaluation + diagnostics defined.")

In [ ]:
# Cell 6 — DSL Mutation + Beam Search (vLLM batched)


def extract_functions_from_response(response):
    functions = []
    blocks = re.findall(r"```python\s*(.*?)(?:\s*```|\Z)", response, re.DOTALL)
    if not blocks:
        blocks = re.findall(r"```\s*(.*?)(?:\s*```|\Z)", response, re.DOTALL)
    for block in blocks:
        parts = re.split(r"(?=\ndef )", block)
        for part in parts:
            part = part.strip()
            if part.startswith("def "):
                healed = _heal_truncated(part)
                if healed:
                    functions.append(healed)
    return functions


def _heal_truncated(func_code):
    try:
        compile(func_code, "<proposed>", "exec")
        return func_code
    except SyntaxError:
        pass
    lines = func_code.split('\n')
    for i in range(len(lines) - 1, 0, -1):
        candidate = '\n'.join(lines[:i]).rstrip()
        if not candidate:
            continue
        try:
            compile(candidate, "<proposed>", "exec")
            return candidate
        except SyntaxError:
            continue
    return None


def validate_function(func_code):
    try:
        compile(func_code, "<proposed>", "exec")
    except SyntaxError:
        return False
    ns = load_dsl_namespace()
    helper_code = ns.get("HELPER_CODE_PREFIX", "")
    func_name = re.match(r"def\s+(\w+)", func_code)
    if func_name and func_name.group(1) in helper_code:
        print(f"  [skip] {func_name.group(1)} already exists in DSL")
        return False
    return True


def inject_functions_into_dsl(new_functions):
    dsl_code = DSL_PATH.read_text()
    closing_marker = "'''"
    last_close = dsl_code.rfind(closing_marker)
    if last_close == -1:
        print("[evolve] ERROR: Cannot find HELPER_CODE_PREFIX closing marker")
        return dsl_code
    injection = "\n\n# --- EVOLVED FUNCTIONS (auto-generated) ---\n"
    for func in new_functions:
        injection += "\n" + func + "\n"
    return dsl_code[:last_close] + injection + "\n" + dsl_code[last_close:]


def log_hypothesis(round_num, hypothesis, mode, proposed_functions, metric_before,
                   metric_after, solve_before, solve_after, status, commit_hash=None):
    entry = {
        "round": round_num, "hypothesis": hypothesis, "mode": mode,
        "proposed_functions": proposed_functions,
        "metric_before": metric_before, "metric_after": metric_after,
        "solve_before": solve_before, "solve_after": solve_after,
        "status": status, "commit_hash": commit_hash,
        "timestamp": datetime.now().isoformat(),
    }
    with open(HYPOTHESES_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")


# --- Inline beam search (no git worktrees on Kaggle) ---

def run_codopt_round(failing_results):
    """Beam search: generate DSL mutations, evaluate, keep best.
    
    Uses vLLM batching for parallel candidate generation.
    No git worktrees — saves/restores dsl.py in memory.
    """
    dsl_backup = DSL_PATH.read_text()
    ns = load_dsl_namespace()
    helper_code = ns.get("HELPER_CODE_PREFIX", "")

    # Build optimization prompt
    sigs = _extract_function_signatures(helper_code)
    ban_list = "\n".join(f"- {s}" for s in sigs[-30:])
    focus_areas = ["symmetry", "object extraction", "pattern tiling",
                   "color mapping", "boundary detection"]
    focus = random.choice(focus_areas)

    opt_prompt = f"""You are optimizing an ARC-AGI DSL. Generate 3-5 NEW helper functions.

## BANNED NAMES (already exist):
{ban_list}

## Focus: {focus}

Output ONLY ```python blocks with function definitions. Each function:
- Takes grid: list[list[int]], returns list[list[int]]
- Self-contained (stdlib + numpy only)
- One-line docstring

```python
def function_name(grid: list[list[int]]) -> list[list[int]]:
    \"\"\"One-line description.\"\"\"
    # implementation
    return result
```"""

    # Batched generation: CODOPT_BRANCHES candidates at once
    temps = [0.4 + 0.15 * i for i in range(CODOPT_BRANCHES)]
    prompts = [opt_prompt] * CODOPT_BRANCHES
    print(f"[beam] Generating {CODOPT_BRANCHES} candidates (batched via vLLM)...")
    responses = _generate_batch(prompts, temperature=0.5)

    # Evaluate each candidate
    baseline_score = _quick_benchmark()
    print(f"[beam] Baseline score: {baseline_score:.4f}")
    best_score = baseline_score
    best_dsl = dsl_backup
    best_id = "baseline"

    for ci, response in enumerate(responses):
        # Restore baseline before each candidate
        DSL_PATH.write_text(dsl_backup)

        raw_functions = extract_functions_from_response(response)
        valid_functions = [f for f in raw_functions if validate_function(f)]

        if not valid_functions:
            print(f"  [candidate {ci}] No valid functions")
            continue

        func_names = [re.match(r"def\s+(\w+)", f).group(1) for f in valid_functions if re.match(r"def\s+(\w+)", f)]
        print(f"  [candidate {ci}] Injecting {len(valid_functions)} functions: {func_names}")

        mutated_dsl = inject_functions_into_dsl(valid_functions)
        DSL_PATH.write_text(mutated_dsl)

        # Quick benchmark: test helpers + solve 2 tasks
        score = _quick_benchmark()
        print(f"  [candidate {ci}] Score: {score:.4f}")

        if score > best_score:
            best_score = score
            best_dsl = mutated_dsl
            best_id = f"candidate_{ci}"

    # Apply best
    DSL_PATH.write_text(best_dsl)
    improved = best_id != "baseline"
    print(f"[beam] Winner: {best_id} (score={best_score:.4f})"
          f"{' [IMPROVED]' if improved else ''}")
    return True, improved, best_score


def _quick_benchmark():
    """Fast benchmark: tier1+tier2 on 20 tasks + tier3 on 2."""
    ns = load_dsl_namespace()
    helper_ns, err = _prepare_helper_ns(ns)
    if helper_ns is None:
        return 0.0

    task_files = list(ARC_DATA.glob("*.json"))
    random.seed(int(time.time()) % 100000)
    sample = random.sample(task_files, min(20, len(task_files)))

    t1_total, t2_total = 0.0, 0.0
    for tf in sample:
        with open(tf) as f:
            td = json.load(f)
        t1_total += eval_task_tier1(helper_ns, td)
        t2_total += eval_task_tier2(ns, td)

    base_score = (t1_total + t2_total) / (2 * len(sample))

    # Tier 3 mini-solve on 2 random tasks
    solve_sample = random.sample(task_files, min(2, len(task_files)))
    solve_total = 0.0
    for tf in solve_sample:
        with open(tf) as f:
            td = json.load(f)
        try:
            sc, _ = solve_task_single(td, task_name=tf.stem)
            solve_total += sc
        except Exception:
            pass

    solve_score = solve_total / len(solve_sample) if solve_sample else 0.0
    blended = base_score * 0.7 + solve_score * 0.3
    return blended


print("DSL mutation + beam search defined.")

In [ ]:
# Cell 7 — LoRA Self-Distillation (PEFT/CUDA, replaces mlx_lm.lora)

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_BATCH_SIZE = 2  # T4x2 can handle batch=2
LORA_MAX_SEQ_LEN = 1024
LORA_EPOCHS = 3
LORA_LR = 2e-5
CANARY_COUNT = 5
CANARY_MIN_SOLVES = 3


def _load_and_dedup_programs(jsonl_path):
    entries, seen = [], set()
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            key = hashlib.md5(f"{entry['task_name']}:{entry['code']}".encode()).hexdigest()
            if key not in seen:
                seen.add(key)
                entries.append(entry)
    return entries


def _format_training_data(entries):
    """Format as chat messages for Qwen tokenizer."""
    formatted = []
    for e in entries:
        formatted.append({
            "messages": [
                {"role": "user", "content": e["prompt"]},
                {"role": "assistant", "content": f"```python\n{e['code']}\n```"},
            ]
        })
    return formatted


def run_lora_training(adapter_output_dir):
    """Train LoRA adapter using PEFT + transformers on CUDA.
    
    Must be called AFTER unload_model() to free GPU memory.
    Returns True if training + validation passed.
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
    from peft import LoraConfig, get_peft_model, TaskType

    if not SUCCESSFUL_PROGRAMS_PATH.exists():
        print("[lora] No successful programs file")
        return False

    # Load and prepare data
    entries = _load_and_dedup_programs(SUCCESSFUL_PROGRAMS_PATH)
    if len(entries) < 10:
        print(f"[lora] Only {len(entries)} unique programs — need at least 10")
        return False

    formatted = _format_training_data(entries)
    random.seed(42)
    random.shuffle(formatted)
    split = max(1, int(len(formatted) * 0.9))
    train_data = formatted[:split]
    val_data = formatted[split:] or formatted[:1]
    print(f"[lora] Data: {len(entries)} unique → {len(train_data)} train, {len(val_data)} val")

    # Load base model for training (not vLLM — raw transformers)
    print(f"[lora] Loading {MODEL_PATH} for training...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

    # Apply LoRA
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Tokenize data
    def tokenize_messages(item):
        text = tokenizer.apply_chat_template(item["messages"], tokenize=False)
        tokens = tokenizer(text, truncation=True, max_length=LORA_MAX_SEQ_LEN,
                          padding="max_length", return_tensors="pt")
        tokens["labels"] = tokens["input_ids"].clone()
        return tokens

    from torch.utils.data import Dataset, DataLoader

    class ChatDataset(Dataset):
        def __init__(self, data):
            self.data = data
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            tokens = tokenize_messages(self.data[idx])
            return {k: v.squeeze(0) for k, v in tokens.items()}

    train_dataset = ChatDataset(train_data)
    train_loader = DataLoader(train_dataset, batch_size=LORA_BATCH_SIZE, shuffle=True)

    # Train
    optimizer = torch.optim.AdamW(model.parameters(), lr=LORA_LR)
    model.train()
    total_loss = 0.0
    steps = 0

    for epoch in range(LORA_EPOCHS):
        for batch in train_loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            steps += 1
            if steps % 20 == 0:
                print(f"  [lora] step {steps} loss={total_loss/steps:.4f}")

    print(f"[lora] Training done: {steps} steps, avg_loss={total_loss/max(steps,1):.4f}")

    # Save adapter
    adapter_output_dir = Path(adapter_output_dir)
    adapter_output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(adapter_output_dir))
    tokenizer.save_pretrained(str(adapter_output_dir))
    print(f"[lora] Adapter saved to {adapter_output_dir}")

    # Canary validation
    task_counts = Counter(e["task_name"] for e in entries)
    reliable = [(name, cnt) for name, cnt in task_counts.items() if cnt >= CANARY_MIN_SOLVES]
    reliable.sort(key=lambda x: -x[1])
    canary_names = [name for name, _ in reliable[:CANARY_COUNT]]
    if not canary_names:
        canary_names = [name for name, _ in sorted(task_counts.items(), key=lambda x: -x[1])[:CANARY_COUNT]]

    all_passed = True
    for task_name in canary_names:
        task_path = ARC_DATA / f"{task_name}.json"
        if not task_path.exists():
            continue
        td = json.loads(task_path.read_text())
        prompt = _fallback_prompt(td)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=1024, temperature=0.01, do_sample=True)
        response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        code = extract_python_code(response)
        passed, _ = try_code_on_task(code, td)
        status = "PASS" if passed else "FAIL"
        print(f"  [canary] {task_name}: {status}")
        if not passed:
            all_passed = False

    # Cleanup training model
    del model, tokenizer, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    return all_passed


# --- LoRA state management ---

def _load_lora_state():
    if LORA_STATE_PATH.exists():
        try:
            return json.loads(LORA_STATE_PATH.read_text())
        except Exception:
            pass
    return {
        "last_training_round": 0, "programs_at_last_training": 0,
        "active_adapter": None, "adapter_history": [], "post_lora_scores": [],
    }


def _save_lora_state(state):
    LORA_STATE_PATH.write_text(json.dumps(state, indent=2))


def _count_successful_programs():
    if not SUCCESSFUL_PROGRAMS_PATH.exists():
        return 0
    with open(SUCCESSFUL_PROGRAMS_PATH) as f:
        return sum(1 for line in f if line.strip())


def should_trigger_lora_training(round_num, state):
    if round_num <= LORA_SKIP_FIRST_ROUNDS:
        return False
    rounds_since = round_num - state.get("last_training_round", 0)
    if rounds_since < LORA_MIN_ROUNDS_BETWEEN:
        return False
    prog_count = _count_successful_programs()
    new_progs = prog_count - state.get("programs_at_last_training", 0)
    return new_progs >= LORA_MIN_PROGRAMS


def run_lora_training_cycle(round_num, state):
    """Full LoRA cycle: unload vLLM → train PEFT → validate → reload vLLM with adapter."""
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    adapter_dir = LORA_ADAPTERS_DIR / timestamp

    print(f"[lora] Starting training cycle (adapter: {timestamp})")
    unload_model()

    passed = False
    try:
        passed = run_lora_training(adapter_dir)
    except Exception as e:
        print(f"[lora] Training error: {e}")
        traceback.print_exc()

    if passed:
        adapter_path = str(adapter_dir)
        reload_with_adapter(adapter_path)
        state["last_training_round"] = round_num
        state["programs_at_last_training"] = _count_successful_programs()
        state["active_adapter"] = adapter_path
        state["adapter_history"].append(timestamp)
        state["post_lora_scores"] = []

        # Prune old adapters
        if len(state["adapter_history"]) > LORA_MAX_KEPT_ADAPTERS:
            import shutil
            old = state["adapter_history"][:-LORA_MAX_KEPT_ADAPTERS]
            for old_ts in old:
                old_dir = LORA_ADAPTERS_DIR / old_ts
                if old_dir.exists():
                    shutil.rmtree(old_dir, ignore_errors=True)
            state["adapter_history"] = state["adapter_history"][-LORA_MAX_KEPT_ADAPTERS:]

        print(f"[lora] Adapter {timestamp} active.")
    else:
        print(f"[lora] Adapter rejected. Reverting to {'previous' if state.get('active_adapter') else 'base model'}.")
        reload_with_adapter(state.get("active_adapter"))

    _save_lora_state(state)
    return state


def check_lora_regression(state, solve_score):
    if not state.get("active_adapter"):
        return state
    scores = state.get("post_lora_scores", [])
    scores.append(solve_score)
    state["post_lora_scores"] = scores
    if len(scores) >= 3 and scores[-1] < scores[-3] and scores[-2] < scores[-3]:
        print(f"[lora] REGRESSION: {scores[-3]:.4f} → {scores[-2]:.4f} → {scores[-1]:.4f}")
        print("[lora] Rolling back to base model")
        reload_with_adapter(None)
        state["active_adapter"] = None
        state["adapter_history"] = []
        state["post_lora_scores"] = []
        _save_lora_state(state)
    return state


print("LoRA self-distillation defined.")

In [ ]:
# Cell 8 — Main Evolution Loop

def evolve(rounds=None, never_stop=False, enable_lora=True):
    """Run the autoresearch evolution loop.
    
    9-step cycle per round:
      0. LORA — self-distillation check
      1. EVALUATE — three-tier task scoring
      2. DIAGNOSE — analyze failures
      3. HYPOTHESIZE — generate DSL functions via vLLM
      4. INJECT — validate + inject into dsl.py
      5. EXPERIMENT — beam search tournament (vLLM batched)
      6. ANALYZE — post-codopt benchmark
      7. DECIDE — keep or revert
      8. RECORD — log hypothesis
    """
    if rounds is None:
        rounds = OUTER_ROUNDS

    lora_state = _load_lora_state() if enable_lora else None

    # Baseline
    print("[evolve] Running baseline benchmark...")
    baseline = _quick_benchmark()
    current_score = baseline
    print(f"[evolve] Baseline score: {current_score:.4f}")

    round_num = 0
    while True:
        round_num += 1
        if not never_stop and round_num > rounds:
            break

        t0 = time.time()
        label = f"{round_num}" + ("" if never_stop else f"/{rounds}")
        print(f"\n{'='*60}")
        print(f"  AUTORESEARCH ROUND {label}")
        print(f"{'='*60}")

        # --- 0. LORA CHECK ---
        if enable_lora and should_trigger_lora_training(round_num, lora_state):
            print(f"\n[0/9] LORA — self-distillation training triggered")
            lora_state = run_lora_training_cycle(round_num, lora_state)
        elif enable_lora:
            prog_count = _count_successful_programs()
            new_progs = prog_count - (lora_state.get("programs_at_last_training", 0))
            rounds_since = round_num - lora_state.get("last_training_round", 0)
            print(f"\n[0/9] LORA — skip (new_programs={new_progs}, rounds_since={rounds_since})")

        # --- 1. EVALUATE ---
        print("\n[1/9] EVALUATE — three-tier task scoring...")
        failing = find_failing_tasks(tier3_count=TIER3_TASKS, seed=round_num)
        if not failing:
            print("[evolve] No failing tasks — DSL may be perfect!")
            if not never_stop:
                break
            failing = find_failing_tasks(tier3_count=TIER3_TASKS,
                                         seed=round_num * 1000 + datetime.now().microsecond)
            if not failing:
                print("[evolve] Still perfect. Sleeping 60s...")
                time.sleep(60)
                continue

        solve_before = compute_solve_score(failing)
        print(f"[evolve] Pre-round solve score: {solve_before:.4f}")

        # --- 2. DIAGNOSE ---
        print(f"\n[2/9] DIAGNOSE — analyzing {len(failing)} failures...")
        diag_prompt = build_diagnostic_prompt(failing)
        diag_path = RESULTS_DIR / f"round{round_num}_diagnostic.txt"
        diag_path.write_text(diag_prompt)

        # --- 3. HYPOTHESIZE ---
        recent = load_recent_hypotheses(10)
        stagnant_streak = _count_stagnant_streak(recent)
        temp = min(0.6 + stagnant_streak * 0.1, 0.9)
        print(f"\n[3/9] HYPOTHESIZE — generating via vLLM (temp={temp:.2f}, stagnant={stagnant_streak})...")

        response = generate(diag_prompt, temperature=temp)
        resp_path = RESULTS_DIR / f"round{round_num}_response.txt"
        resp_path.write_text(response)
        print(f"[evolve] Response ({len(response)} chars) saved")

        # --- 4. INJECT ---
        print("\n[4/9] INJECT — extracting and validating functions...")
        new_functions = extract_functions_from_response(response)
        validated = [f for f in new_functions if validate_function(f)]
        func_names = []
        for v in validated:
            m = re.match(r"def\s+(\w+)", v)
            if m:
                func_names.append(m.group(1))
        print(f"[evolve] Proposed {len(new_functions)}, valid {len(validated)}: {func_names}")
        hypothesis = f"Add {', '.join(func_names)}" if func_names else "No valid functions proposed"

        if validated:
            mutated_dsl = inject_functions_into_dsl(validated)
            DSL_PATH.write_text(mutated_dsl)
            print(f"[evolve] Injected {len(validated)} functions into dsl.py")

        # --- 5. EXPERIMENT ---
        print("\n[5/9] EXPERIMENT — running beam search tournament...")
        pre_score = current_score
        codopt_ok, beam_improved, beam_score = run_codopt_round(failing)

        # --- 6. ANALYZE ---
        print("\n[6/9] ANALYZE — post-codopt benchmark...")
        post_score = _quick_benchmark()
        solve_after = beam_score if codopt_ok else 0.0

        print(f"[evolve] Results: metric {pre_score:.4f}->{post_score:.4f}, "
              f"solve {solve_before:.4f}->{solve_after:.4f}")

        # --- 7. DECIDE ---
        if post_score >= 0.999 and beam_improved:
            improved = True
            status = "keep"
        else:
            improved = post_score > pre_score
            status = "keep" if improved else "stagnant"

        if not improved:
            # Revert dsl.py to pre-round state
            print(f"\n[7/9] DECIDE — {status}, reverting dsl.py")
        else:
            current_score = post_score
            print(f"\n[7/9] DECIDE — {status} (score {pre_score:.4f}->{post_score:.4f})")

        # --- 8. RECORD ---
        print(f"\n[8/9] RECORD")
        log_hypothesis(
            round_num, hypothesis, "diagnostic", func_names,
            pre_score, post_score, solve_before, solve_after, status,
        )

        # --- LoRA regression check ---
        if enable_lora and lora_state:
            lora_state = check_lora_regression(lora_state, solve_after)

        elapsed = time.time() - t0
        print(f"\n[evolve] Round {round_num} done. Score: {current_score:.4f} [{status}] ({elapsed:.0f}s)")

    print(f"\n{'='*60}")
    print(f"  EVOLUTION COMPLETE — Final Score: {current_score:.4f}")
    print(f"  Successful programs logged: {_count_successful_programs()}")
    print(f"{'='*60}")

    # Save final dsl.py as artifact
    final_dsl = RESULTS_DIR / "dsl_final.py"
    import shutil
    shutil.copy(DSL_PATH, final_dsl)
    print(f"Final DSL saved to {final_dsl}")


print("Evolution loop defined.")

In [ ]:
# Cell 9 — Run Evolution Loop
#
# T4x2 budget: ~12 hours Kaggle runtime
# ~20 rounds × ~30 min/round = ~10 hours + margin for LoRA training
#
# Set never_stop=True to use full 12h budget
# Set enable_lora=False to skip LoRA (simpler, faster)

evolve(rounds=20, never_stop=False, enable_lora=True)